In [ ]:
!pip install pandas numpy scikit-learn tensorflow transformers matplotlib seaborn nltk



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd

data = pd.read_csv("dataset.csv")  # adjust filepath
print(data.head())
print(data.columns)
print(data.shape)


   Unnamed: 0.1  Unnamed: 0  \
0             0           0   
1             1           1   
2             2           2   
3             3           3   
4             4           4   

                                                Text    Sentiment  \
0   Enjoying a beautiful day at the park!        ...   Positive     
1   Traffic was terrible this morning.           ...   Negative     
2   Just finished an amazing workout! 💪          ...   Positive     
3   Excited about the upcoming weekend getaway!  ...   Positive     
4   Trying out a new recipe for dinner tonight.  ...   Neutral      

             Timestamp            User     Platform  \
0  2023-01-15 12:30:00   User123          Twitter     
1  2023-01-15 08:45:00   CommuterX        Twitter     
2  2023-01-15 15:45:00   FitnessFan      Instagram    
3  2023-01-15 18:20:00   AdventureX       Facebook    
4  2023-01-15 19:55:00   ChefCook        Instagram    

                                     Hashtags  Retweets  Likes     

In [9]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop = stopwords.words('english')

clean_texts = []
for txt in data['Text']:  # Corrected column name from 'text' to 'Text'
    txt = str(txt).lower()
    txt = re.sub(r"http\S+", "", txt)
    txt = re.sub(r"[^a-z\s]", "", txt)
    filtered = []
    for word in txt.split():
        if word not in stop:
            filtered.append(word)
    clean_texts.append(" ".join(filtered))

data['clean_text'] = clean_texts

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\venka\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
#3) Feature Extraction — TF-IDF & Word Embeddings
# A. TF-IDF Vectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)
X_tfidf = tfidf.fit_transform(data['clean_text'])


In [ ]:
# BERT embeddings (transformer based model implementation)

from transformers import BertTokenizer, TFBertModel
import tensorflow as tf

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = TFBertModel.from_pretrained('bert-base-uncased', use_safetensors=False)

input_ids = []
attention_masks = []

for text in data['clean_text']:
    encoded = tokenizer.encode_plus(text, max_length=128, padding='max_length',
                                     return_attention_mask=True, truncation=True, return_tensors='tf')
    input_ids.append(encoded['input_ids'])
    attention_masks.append(encoded['attention_mask'])

input_ids = tf.concat(input_ids, axis=0)
attention_masks = tf.concat(attention_masks, axis=0)

bert_outputs = bert(input_ids, attention_mask=attention_masks)
X_bert = bert_outputs.last_hidden_state[:,0,:].numpy()

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [ ]:
#create labels for supervised models

# Define a list of sentiments to be considered 'positive'
positive_sentiments = ['Positive', 'Joy', 'Excitement', 'Happy', 'Good', 'Great', 'Amazing', 'Wonderful', 'Fantastic', 'Lovely', 'Beautiful',
                       'Vibrancy', 'Culinary Adventure', 'Mesmerizing', 'Thrilling Journey', 'Winter Magic', 'Elation', 'Love', 'Gratitude', 'Optimistic', 'Contentment', 'Peaceful', 'Hopeful']

# Clean 'Sentiment' column by stripping spaces and capitalizing
data['Sentiment_cleaned'] = data['Sentiment'].astype(str).str.strip().str.capitalize()

# Convert 'Sentiment' column to binary labels: 1 for positive, 0 for others
y = data['Sentiment_cleaned'].apply(lambda x: 1 if x in positive_sentiments else 0)

print(y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X_train_tf, X_test_tf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
X_train_b, X_test_b, _, _ = train_test_split(X_bert, y, test_size=0.2, random_state=42)

In [ ]:
#logistic regression

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tf, y_train)

print("Train Accuracy:", lr.score(X_train_tf, y_train))
print("Test Accuracy:", lr.score(X_test_tf, y_test))

In [ ]:
#random forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300)
rf.fit(X_train_tf, y_train)

print("RF Test Accuracy:", rf.score(X_test_tf, y_test))

In [ ]:
# deep learning classifier (BERT embeddings)
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(768,)),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train_b, y_train, epochs=5, batch_size=32, validation_split=0.2)
model.evaluate(X_test_b, y_test)

In [ ]:
print(data['Sentiment'].value_counts())

In [ ]:
print(data['Sentiment'].value_counts())

In [ ]:
#k means clustering with  tfidf
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=10, random_state=42)
kmeans.fit(X_tfidf)

data['cluster'] = kmeans.labels_
print(data['cluster'].value_counts())


In [ ]:
#lda(latent dirichlet allocation)

from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=10)
lda.fit(X_tfidf)

topic_results = lda.transform(X_tfidf)


In [ ]:
#visual insights and trends


import matplotlib.pyplot as plt
import seaborn as sns

# sentiment distribution
sns.countplot(x=data['Sentiment_cleaned']) # Corrected column name
plt.show()

# cluster distribution
sns.countplot(x=data['cluster'])
plt.title("Cluster Count")
plt.show()

In [ ]:
# advanced insights : sentiments vs engagement
sns.boxplot(x=data['Sentiment_cleaned'], y=data['Likes'])
plt.title("Likes by Sentiment")
plt.show()

sns.boxplot(x=data['Sentiment_cleaned'], y=data['Retweets'])
plt.title("Retweets by Sentiment")
plt.show()

In [ ]:
# temporal trend

data['timestamp'] = pd.to_datetime(data['Timestamp'])
data['date'] = data['timestamp'].dt.date

daily = data.groupby('date').agg({'Likes':'sum','Retweets':'sum'}).reset_index()
plt.plot(daily['date'], daily['Likes'])
plt.title("Daily Likes Over Time")
plt.show()

In [ ]:
import pickle

pickle.dump(lr, open("model.pkl", "wb"))
pickle.dump(tfidf, open("vectorizer.pkl", "wb"))


In [ ]:
from flask import Flask, render_template, request
import pandas as pd
import pickle

app = Flask(__name__)

model = pickle.load(open("model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    file = request.files["file"]
    data = pd.read_csv(file)

    texts = data['clean_text']
    vectors = vectorizer.transform(texts)

    predictions = model.predict(vectors)
    data['Predicted_Sentiment'] = predictions

    positive = (data['Predicted_Sentiment'] == 1).sum()
    negative = (data['Predicted_Sentiment'] == 0).sum()

    return render_template(
        "result.html",
        tables=[data.head(20).to_html(classes='data')],
        positive=positive,
        negative=negative
    )

if __name__ == "__main__":
    app.run(debug=False)
